# 08b — Capstone Upgrade: `AsyncSqliteSaver` + `interrupt()` + Time Travel

Notebook 08 built a tier-2 HITL gate **by hand** — `ApprovalStore` with a `pending/` directory of JSON files and a 500ms polling loop. It works. It's also exactly the kind of wheel LangGraph reinvented properly.

This notebook replaces all of it with **three LangGraph primitives**:

1. **Checkpointer** (`AsyncSqliteSaver` — sqlite-backed, async-friendly) — every superstep's state survives process restart
2. **`interrupt()`** — durable HITL inside a node, no polling, no approval table
3. **Time travel** — `app.get_state_history(config)` returns every checkpoint; fork + replay any of them

**By the end of this notebook you will:**
1. Run a research pipeline that pauses at `interrupt()` and inspect what's preserved
2. **Kill the Python app object mid-run**, recreate it from scratch, and resume — proving the state is durable
3. Resume with `Command(resume=...)` and watch the graph continue from exactly where it paused
4. Use `get_state_history` to reproduce the agent's behavior and fork from an earlier point with modified state

Lecture references: **S9 §3.1** (Checkpointer), **S9 §3.2** (`interrupt()`), **S9 §3.3** (Time travel).


## 1. Why the hand-rolled `ApprovalStore` isn't enough

Notebook 08's flow:

```
draft ready → ApprovalStore.create() → file in pending/  
  Python kernel sits in `await wait_for_decision()` polling every 500ms  
  ipywidgets button click → ApprovalStore.decide() → file moves to decided/  
  poll loop sees state change → returns → save brief
```

Four limitations:

| What hurts | Why |
|---|---|
| The Python process must stay alive | Polling loop holds the await. Worker recycle = brief abandoned. |
| Only the approval state is saved | All the in-flight research state lives in `agent.run()` locals. Crash mid-run = restart from scratch. |
| No way to inspect what the agent did at each step | The trace is in memory. After the cell finishes, it's gone. |
| 500ms polling burns CPU + adds latency | Acceptable in a notebook. Wasteful at fleet scale. |

LangGraph fixes all four with one wiring change. We don't write more code — we write **different** code.


## 2. Setup

In [ ]:
import asyncio
import os
import tempfile
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

# ⚠️ Use the *async* SqliteSaver — our nodes are async, sync SqliteSaver
# raises NotImplementedError on aget_tuple() the first time the graph reads state.
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.types import Command

# We use a tempfile-backed sqlite db so this notebook is hermetic.
# In production: use AsyncPostgresSaver pointing at a real Postgres.
DB_PATH = str(Path(tempfile.gettempdir()) / "deepbrief_capstone.db")
print(f"checkpointer db: {DB_PATH}")

## 3. The new graph: `langgraph_capstone.py`

We've added one node — `review` — to the graph from notebook 07b. The whole capstone source lives at `src/deepbrief/agents/langgraph_capstone.py`.

In [ ]:
import inspect
from deepbrief.agents.langgraph_capstone import (
    CapstoneState, review_node, needs_revision,
    build_capstone_graph, build_capstone_app, empty_state,
)

print("=== CapstoneState ===")
print(inspect.getsource(CapstoneState))

print("=== review_node (where interrupt() lives) ===")
print(inspect.getsource(review_node))

**The whole new pattern is in `review_node`.** Five things to call out (S9 §3.2):

**(a) `interrupt(payload)` is a function call inside the node.** Not a config flag, not a decorator. The node decides when to pause — based on state. You can interrupt unconditionally for tier-2 actions or conditionally on risk score.

**(b) The payload is by convention.** Any JSON-serializable dict. Surface it to a UI; the UI's schema is yours to define.

**(c) `interrupt(...)` literally returns the resume value.** When the human resumes via `Command(resume="approve")`, the *same line* of code that initially called `interrupt(...)` now evaluates to `"approve"` and execution continues. The graph rehydrates and walks forward.

**(d) `MAX_REVISIONS = 2` is our termination guard.** Without it the reviewer + LLM can bounce forever — `synthesize → review → synthesize → review → ...`. Map this to S8's loop-cap discipline.

**(e) `needs_revision` is the conditional edge.** It reads `approved` and `reviewer_notes` and decides END or back-to-synthesize.


## 4. Build the app — checkpointer is required

Two-step compile:

In [ ]:
# AsyncSqliteSaver.from_conn_string is an **async** context manager.
# We use AsyncExitStack so we can keep it open across multiple notebook cells
# while still cleaning up correctly at the end.
from contextlib import AsyncExitStack

stack = AsyncExitStack()
checkpointer = await stack.enter_async_context(AsyncSqliteSaver.from_conn_string(DB_PATH))
app = build_capstone_app(checkpointer)
print("app compiled with AsyncSqliteSaver — interrupt() will work")

**Why is the checkpointer required?**

`interrupt()` writes state to durable storage and returns control to the caller. If there's no checkpointer, there's nowhere to write — the call would lose all state. LangGraph guards against this: `build_capstone_app(None)` raises `ValueError`.

**`thread_id` is the unit of state isolation.** Each thread has its own checkpoint history. Multi-tenant systems share one graph across millions of conversations by partitioning on `thread_id` — same schema, isolated data.

## 5. First call — runs until `interrupt()`

Same topic as notebook 07b for direct comparison. The call returns when the graph hits `review`.

In [ ]:
TOPIC = "State of agent-protocol adoption: MCP vs A2A in 2026"
THREAD_ID = "capstone-demo-1"
config = {"configurable": {"thread_id": THREAD_ID}}

# Run until the graph pauses
result = await app.ainvoke(empty_state(TOPIC), config=config)

print("=== top-level result keys ===")
for k, v in result.items():
    if k == "__interrupt__":
        print(f"  {k}: {v}")
    elif isinstance(v, (str, int, float, bool)) or v is None:
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: <{type(v).__name__} of len {len(v)}>")

Notice the **`__interrupt__`** key — that's how LangGraph tells the caller "the graph paused." The payload contains exactly what we passed to `interrupt(...)`.

Also notice that **`draft` is populated** even though `approved` is still `False`. The graph got all the way through `synthesize`, paused inside `review`, and surfaced the partial state back to us. **All the research work is done and persisted** — the only thing waiting is the human.

## 6. The dramatic move: kill the app, recreate it, prove durability

Here's the test that the hand-rolled `ApprovalStore` could pass too, but `SqliteSaver` passes **without us writing any save/load code**. We're going to:

1. Delete the compiled `app` object
2. Wait (simulate a process restart — but in-notebook it's instant)
3. Rebuild a brand new app from scratch
4. Connect to the **same** sqlite db + thread_id
5. Inspect the state — it should be sitting at the `review` interrupt


In [ ]:
# Drop the old app reference (async cleanup)
del app, checkpointer
await stack.aclose()
print("Old app + checkpointer destroyed.\n")

# Recreate from scratch — same DB, same thread_id
stack = AsyncExitStack()
new_checkpointer = await stack.enter_async_context(AsyncSqliteSaver.from_conn_string(DB_PATH))
app = build_capstone_app(new_checkpointer)
print("New app compiled. Reading state from sqlite...\n")

# Use the async variant — our checkpointer is async-only
snapshot = await app.aget_state(config)
print(f"  current values keys:   {list(snapshot.values.keys())}")
print(f"  next node to execute:  {snapshot.next}")
print(f"  __interrupt__ pending? {bool(snapshot.tasks)}")
print(f"  topic:                 {snapshot.values['topic']}")
print(f"  findings count:        {len(snapshot.values['findings'])}")
print(f"  draft length:          {len(snapshot.values['draft'] or '')}")

**Read the output carefully.** A brand new Python object — different memory, no shared state with the original `app` — has just loaded the **complete in-flight state** from a sqlite file, including:

- The topic
- All N findings (the parallel research output)
- The synthesized draft
- The pending interrupt (the `next: ('review',)` confirms the graph is paused at review)

**This is the production property the hand-rolled `ApprovalStore` doesn't give you.** ApprovalStore saved the *approval request*. SqliteSaver saves the *entire agent state*.


## 7. Resume with `Command(resume=...)`

Now we feed a decision back into the paused `interrupt()` call. The graph rehydrates from the checkpoint, the `interrupt(...)` call returns our `value`, and `review_node` continues with that decision.

In [ ]:
# Approve the brief — the simplest case
final = await app.ainvoke(Command(resume="approve"), config=config)

print("=== final state after resume ===")
print(f"  approved:        {final['approved']}")
print(f"  reviewer_notes:  {final.get('reviewer_notes')}")
print(f"  revision_count:  {final.get('revision_count', 0)}")
print(f"  __interrupt__:   {final.get('__interrupt__', None)}")

print("\n=== final brief ===")
print(final["draft"])

## 8. The revision loop — resume with `revise` instead of `approve`

What if the reviewer doesn't like the draft? `Command(resume={"action": "revise", "notes": "..."})` loops back to `synthesize`, which re-runs with the same `findings` (we don't re-research — we only re-synthesize). After `MAX_REVISIONS = 2` rounds, `review_node` auto-approves to prevent infinite loops.

Let's prove it on a fresh thread.

In [ ]:
# Fresh thread for the revision-loop demo
REVISE_THREAD = "capstone-revise-demo"
revise_config = {"configurable": {"thread_id": REVISE_THREAD}}

result = await app.ainvoke(empty_state("What is FastMCP?"), config=revise_config)
print("first pause — draft v1 ready, asking for review")

# Reviewer asks for a revision
result = await app.ainvoke(
    Command(resume={"action": "revise", "notes": "too technical — add a 1-sentence summary at the top"}),
    config=revise_config,
)
print("second pause — draft v2 ready")
print(f"  revision_count now: {result.get('revision_count', 0)}")

# Approve the second draft
result = await app.ainvoke(Command(resume="approve"), config=revise_config)
print("done")
print(f"  approved={result['approved']}  revision_count={result.get('revision_count', 0)}")
print(f"\nfinal draft (length={len(result['draft'])}):")
print(result['draft'][:400] + ("..." if len(result['draft']) > 400 else ""))

Two things just happened that the hand-rolled `ApprovalStore` couldn't do cleanly:

**(a) State preservation across the revision round.** The `findings` list survived the synthesize → review → synthesize cycle untouched — same researchers' output, only the LLM re-merges with the reviewer's notes. With `ApprovalStore` you'd have to re-run the whole pipeline or write custom save-and-restore for the intermediate state.

**(b) The reviewer's structured payload was passed through.** `Command(resume={"action": "revise", "notes": "..."})` — the dict is returned by `interrupt(...)` exactly as-is. The notes can flow into the next synthesize step's prompt if you want — full editorial control.


## 9. Time travel — inspect every checkpoint

**The senior-grade superpower** (S9 §3.3): `app.get_state_history(config)` returns every checkpoint ever taken for this thread. You can inspect them, fork from any of them, replay.

This is the answer to the interview question *"a user reports the agent gave a different answer this time — how do you reproduce?"*

In [ ]:
# List every checkpoint for the approval thread (the first demo, not the revise demo)
# `aget_state_history` is an async iterator — collect into a list
snapshots = [s async for s in app.aget_state_history(config)]
print(f"Total snapshots: {len(snapshots)}\n")

for i, snap in enumerate(snapshots):
    n_findings = len(snap.values.get("findings", []))
    has_draft = bool(snap.values.get("draft"))
    approved = snap.values.get("approved", False)
    next_node = snap.next or ("END",)
    print(f"  [{i:2d}] next={str(next_node):20s}  findings={n_findings}  draft={'yes' if has_draft else 'no '}  approved={approved}")

Each row is a snapshot taken after one superstep. You can see the state evolving — no findings → 1 finding → 2 → 3 → draft appears → approved → done.

**This trace is free.** You did not write a single `trace.append()` call. Compare with S8's hand-rolled `trace = []` list — same information, but now durable and queryable.

## 10. Time travel — fork from a prior checkpoint with modified state

Pick a snapshot from before `synthesize` ran. Modify `findings` (pretend a researcher returned different data). Replay from that point. The graph re-runs only the affected downstream nodes.

In [ ]:
# Find the snapshot right after research_node ran (so we can modify findings before synthesize)
pre_synth = next(
    s for s in snapshots
    if s.next == ("synthesize",)
)
print("forking from this snapshot:")
print(f"  next = {pre_synth.next}")
print(f"  findings count = {len(pre_synth.values['findings'])}")
print(f"  checkpoint_id  = {pre_synth.config['configurable']['checkpoint_id']}")

# Build a fork config pointing at that exact checkpoint
fork_config = {
    "configurable": {
        "thread_id": THREAD_ID,
        "checkpoint_id": pre_synth.config["configurable"]["checkpoint_id"],
    }
}

# Mutate state at the fork point — replace findings with a doctored version
doctored_findings = list(pre_synth.values["findings"])
if doctored_findings:
    doctored_findings[0] = {
        **doctored_findings[0],
        "summary": "[DOCTORED FOR TIME-TRAVEL DEMO] First finding now says MCP was created by the Linux Foundation in 2024 — wrong on purpose.",
    }

# aupdate_state writes a new checkpoint at the fork point (async variant)
await app.aupdate_state(fork_config, {"findings": doctored_findings, "draft": None})

# Re-invoke from the fork — synthesize re-runs with the doctored findings
forked = await app.ainvoke(None, config=fork_config)
print("\n=== forked draft (should reflect the doctored finding) ===")
print(forked.get("draft", "<paused>")[:500])

If everything worked, the forked draft's text differs from the original — because synthesize ran against the doctored findings. **You just A/B-tested the synthesize step with surgical input control**, without re-running the expensive research nodes.

**This is the time-travel pattern** — fork → update_state → ainvoke. Three lines, the entire reproducibility story.

Real worked example: see [DEV.to time-travel debugging a loan-rejection agent](https://dev.to). The pattern generalizes — banking, compliance, any agent where you need to defend a decision.


## 11. Mapping back to S8's HITL 4-tier taxonomy

S8 §5.5 said: *"This taxonomy is not encoded per-agent. It is a fleet-wide policy."* LangGraph gives us the primitives to implement each tier consistently:

| Tier | Risk | Lecture mode | LangGraph implementation |
|---|---|---|---|
| **0** | Read-only, idempotent | Auto + log only | No `interrupt`; just trace via `astream_events` |
| **1** | Non-critical writes | Logged async | No `interrupt`; emit event to a reviewer queue (Redis pub/sub) |
| **2** | Irreversible writes / customer-facing | **Sync HITL** | `interrupt()` — exactly what we just built |
| **3** | Regulated / high-stakes | Hard block | Don't propose; refuse in the planner |

The neat property: **all four tiers share the same agent code base** — only the node-level policy decides which tier applies. New action type? Tag it with a tier; the runtime takes care of the rest. Fleet-wide consistency for free.


## 12. Cleanup

In [ ]:
await stack.aclose()
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"removed {DB_PATH}")

## 13. Self-check

1. Why does `interrupt()` require a checkpointer? What would go wrong without one?
2. We killed the `app` object mid-run and recreated it. What did the new app object inherit from the old? Where was the state physically stored?
3. `Command(resume=...)` can be a string, a dict, anything JSON-serializable. What's the architectural reason to use a dict over a string in production?
4. We capped revisions with `MAX_REVISIONS = 2`. What's the runaway scenario this guards against?
5. Walk through what `app.update_state(fork_config, {"findings": doctored})` actually does. Where does the doctored data physically end up?

## What's next

Notebook **09 — production pipeline**. We've got an agent that's durable and interruptable. Now we need to run it as a **worker consuming from a queue**, with idempotency, progress emission, and durable result storage. That's S9 Part 5 — the operational scaffolding around the agent. It maps line-by-line to the lecture.
